In [26]:
import geopandas as gpd
import json
import os
import pdal
from shapely.geometry import mapping

# Carregar feições de favelas
gdf = gpd.read_file('data/SIRGAS_GPKG_favela.gpkg')
gdf.geometry = gdf.buffer(0)

# Filtrar apenas a São Remo
gdf_sao_remo = gdf.loc[gdf.fv_nome == 'São Remo', :]

# Dicionário com anos, pastas e articulações
als = {
    2017: {
        "arquivos_laz": "../LiDAR-Sampa-2017",
        "articulacao": "data/quadricula_folha_mdt.zip",
        "atributo_quadricula": "cd_quadric",
        "nome_fn": lambda row, dados: f"MDS_color_{row[dados['atributo_quadricula']]}.laz"
    },
    2020: {
        "arquivos_laz": "../LiDAR-Sampa-2020",
        "articulacao": "data/quadricula_folha_mdt_mds_2020.zip",
        "atributo_quadricula": "cd_quadric",
        "nome_fn": lambda row, dados: f"MDS_{row[dados['atributo_quadricula']]}_1000.laz"
    },
    2024: {
        "arquivos_laz": "../LiDAR-Sampa-2024",
        "articulacao": "../LiDAR-Sampa-2024/poligonos_consolidados.gpkg",
        "atributo_quadricula": "nome_arquivo",
        "nome_fn": lambda row, dados: f'{row[dados["atributo_quadricula"]]}.laz' 
    }
}

# Pasta de saída geral
os.makedirs("saidas", exist_ok=True)

for ano, dados in als.items():
    print(f"\n=== Processando {ano} ===")
    gdf_art = gpd.read_file(dados['articulacao'])
    gdf_art = gdf_art.to_crs(gdf_sao_remo.crs)

    # Recorte das feições que caem na São Remo
    recorte = gpd.overlay(gdf_art, gdf_sao_remo, how="intersection")
    als[ano]["gdf"] = recorte

    print(f"{ano}: {len(recorte)} feições sobrepostas")

    # Pasta de saída por ano
    outdir = os.path.join("saidas", str(ano))
    os.makedirs(outdir, exist_ok=True)

    arquivos_saida = []

    for idx, row in recorte.iterrows():
        # Ajuste: troque 'NOME' pelo campo que identifica a folha
        nome_folha = dados["nome_fn"](row, dados)   # aqui aplica a função definida no dicionário
        arquivo_laz = os.path.join(dados["arquivos_laz"], nome_folha)
        print(arquivo_laz)

        if not os.path.exists(arquivo_laz):
            print(f"⚠️ Arquivo não encontrado: {arquivo_laz}")
            continue

        # Converter geometria em GeoJSON para o filtro do PDAL
        polygon = mapping(row.geometry)
        out_laz = os.path.join(outdir, f"{nome_folha}_recorte.laz")
        arquivos_saida.append(out_laz)

        pipeline_json = {
            "pipeline": [
                arquivo_laz,
                {
                    "type": "filters.crop",
                    "polygon": json.dumps(polygon)
                },
                {
                    "type": "filters.hag_nn"
                },
                {
                    "type": "filters.ferry",
                    "dimensions": "Z => Z_orig"
                },
                {
                    "type": "writers.las",
                    "filename": out_laz,
                    "extra_dims": "HeightAboveGround=float32,Z_orig=float32"
                }
            ]
        }

                   
        pipeline = pdal.Pipeline(json.dumps(pipeline_json))
        pipeline.execute()
        print(f"✅ Recorte salvo: {out_laz}")

    # Juntar os recortes em um único arquivo do ano
    if arquivos_saida:
        merged_out = os.path.join(outdir, f"SaoRemo_{ano}.laz")
        pipeline_merge = {
            "pipeline": arquivos_saida + [
                {
                    "type": "writers.las",
                    "filename": merged_out,
                    "extra_dims": "HeightAboveGround=float32,Z_orig=float32"
                }
            ]
        }
        pdal.Pipeline(json.dumps(pipeline_merge)).execute()
        print(f"🎯 Nuvem final salva: {merged_out}")
    else:
        print(f"⚠️ Nenhuma nuvem processada para {ano}")


/Users/fernandogomes/miniconda3/envs/pdal/lib/python3.12/site-packages/pyogrio/raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Polygon' is converted to 'Polygon Z'
  return ogr_read(



=== Processando 2017 ===
2017: 4 feições sobrepostas
../LiDAR-Sampa-2017/MDS_color_3313-144.laz
✅ Recorte salvo: saidas/2017/MDS_color_3313-144.laz_recorte.laz
../LiDAR-Sampa-2017/MDS_color_3313-143.laz
✅ Recorte salvo: saidas/2017/MDS_color_3313-143.laz_recorte.laz
../LiDAR-Sampa-2017/MDS_color_3313-312.laz
✅ Recorte salvo: saidas/2017/MDS_color_3313-312.laz_recorte.laz
../LiDAR-Sampa-2017/MDS_color_3313-311.laz
✅ Recorte salvo: saidas/2017/MDS_color_3313-311.laz_recorte.laz
🎯 Nuvem final salva: saidas/2017/SaoRemo_2017.laz

=== Processando 2020 ===
2020: 4 feições sobrepostas
../LiDAR-Sampa-2020/MDS_3313-144_1000.laz
✅ Recorte salvo: saidas/2020/MDS_3313-144_1000.laz_recorte.laz
../LiDAR-Sampa-2020/MDS_3313-143_1000.laz
✅ Recorte salvo: saidas/2020/MDS_3313-143_1000.laz_recorte.laz
../LiDAR-Sampa-2020/MDS_3313-312_1000.laz
✅ Recorte salvo: saidas/2020/MDS_3313-312_1000.laz_recorte.laz
../LiDAR-Sampa-2020/MDS_3313-311_1000.laz
✅ Recorte salvo: saidas/2020/MDS_3313-311_1000.laz_recort